In [ ]:
import sys, pathlib
root = pathlib.Path.cwd()
if not (root / "src").is_dir():          # walk up until we find src/
    root = next(p for p in root.parents if (p / "src").is_dir())
sys.path.insert(0, str(root / "src"))    # src-layout: the package lives under src/

import pandas as pd
import stage_discharge as sd

In [ ]:
tests = root / "imports" / "tests"

# VuSitu now REQUIRES pressure_kind (In-Situ labels both channels "Pressure").
# clean_* returns (data, units): UTC index + SI units (psi->Pa, ft/cm->m, F->C).
_, data_abs, units_abs = sd.read_vusitu_log(
    tests / "VuSitu_Log_2025-10-15_18-00-00_dolan_xing_tube_20251015_dolanXing.csv",
    pressure_kind="absolute",          # in-water sonde
)
data_abs, units_abs = sd.clean_vusitu_log(data_abs, units_abs)
print(units_abs)
print(data_abs.head())

_, data_baro, units_baro = sd.read_vusitu_log(
    tests / "VuSitu_Log_2025-10-15_18-00-00_Dolan_Baro_20251015_DolanBaro.csv",
    pressure_kind="barometric",        # BaroTROLL in air
)
data_baro, units_baro = sd.clean_vusitu_log(data_baro, units_baro)
print(units_baro)
print(data_baro.head())

data_hobo, units_hobo = sd.read_hobo_log(
    tests / "DEV300_20260914_export3-2026_09_14_14_16_02_UTC.csv"
)
data_hobo, units_hobo = sd.clean_hobo_log(data_hobo, units_hobo)
print(units_hobo)
print(data_hobo.head())

In [ ]:
# all three readers land on one UTC clock
data_abs.index

In [ ]:
# abs_pressure_pa and baro_pressure_pa land on distinct names; temperature_c
# collides, so it gets the _abs / _baro suffixes.
vudolan = pd.merge_asof(
    data_abs.sort_index(), data_baro.sort_index(),
    left_index=True, right_index=True,
    direction="nearest",
    tolerance=pd.Timedelta("1s"),      # pair a reading only with a baro sample within 1 s
    suffixes=("_abs", "_baro"),
)
vudolan.head()

In [ ]:
import matplotlib.pyplot as plt

# build_stage requires SI + UTC (both guaranteed by clean_*). It derives
# diff_pressure_pa from abs - baro and computes water_level_m. Temperature
# collided in the merge, so name the in-water sonde's column explicitly.
stage = sd.build_stage(vudolan, temp_col="temperature_c_abs")
print(stage[["diff_pressure_pa", "water_level_m"]].head())

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(stage.index, stage["water_level_m"])
ax.set_xlabel("time (UTC)")
ax.set_ylabel("water level (m)")
ax.set_title("Dolan crossing — VuSitu stage")
plt.tight_layout()
plt.show()